In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random as rand
import math as math

In [ ]:
def expected_score(A, B):
    rA = A.rating + rand.uniform(-1,1) * A.volatility + 35
    rB = B.rating + rand.uniform(-1,1) * B.volatility
    return 1 / (1 + 10 ** ((rB - rA) / 400))

def draw_chance(A,B):
    rA = A.rating + rand.uniform(-1,1) * A.volatility + 35
    rB = B.rating + rand.uniform(-1,1) * B.volatility
    return max(0.35, 0.5 - abs(rA-rB)/800) 

In [ ]:
playersList = []
winnersList = []
tournamentWinnersList = []
scoresList = []
tiebreaksList = []

In [ ]:
class Player: 
    def __init__(self, name, rating, volatility = 30):

        self.init_rating = rating
        self.name = name
        self.rating = rating
        self.volatility = volatility

        self.gameResults = [] # results of individual games
        self.tournamentResults = [] # results of individual tournaments
        self.winList = [] # opponents won against
        self.lossList = [] # opponents lost against
        self.drawList = [] # opponents drew against
        self.opponentsList = [] # opponents
        self.colorsList = [] # colors vs opponents

        self.tiebreak = 0
        self.wins = 0
        self.losses = 0
        self.draws = 0
        self.score = 0.0
        playersList.append(self)
        self.index = len(playersList) - 1
        tournamentWinnersList.append(0)

    def resetAll(self):
        self.wins = 0
        self.losses = 0
        self.draws = 0
        self.tiebreak = 0
        self.score = 0
        self.rating=self.init_rating
        
        self.gameResults.clear()
        self.winList.clear()
        self.lossList.clear()
        self.drawList.clear()
        self.opponentsList.clear()
        self.colorsList.clear()
        

    def setScore(self):
        self.score = self.wins + (1/2)*self.draws

    def getRecordString(self):
       return f"{self.wins} - {self.losses} - {self.draws}"
    
    def getScoreString(self):
        return f"{self.score}/{float(self.wins + self.losses + self.draws)}"
    
    def calculateTiebreak(self):
        wins = self.winList
        draws = self.drawList
        if len(wins) > 0:
            for opp in wins:
                self.tiebreak += opp.score
        if len(draws) > 0:
            for opp in draws:
                self.tiebreak += (opp.score * 0.5)

In [ ]:

P1 = Player("Wesley", 2754)
P2 = Player("Gukesh", 2717)
P3 = Player("Magnus", 2840)
P4 = Player("Pragg", 2733)
P5 = Player("Vincent", 2759)
P6 = Player("Alireza", 2759)
P7 = Player("Hikaru", 2800)
P8 = Player("Fabi", 2790)

# P1 = Player("Niko", 2634)
# P2 = Player("Mukhiddin", 2586)
# P3 = Player("Shak", 2717)
# P4 = Player("Yakubbeov", 2689)
# P5 = Player("Vokhidov", 2637)
# P6 = Player("Abdusattorov", 2777)
# P7 = Player("Arjun", 2761)
# P8 = Player("Vidit", 2708)
# P9 = Player("Hans", 2742)
# P10 = Player("Nepo", 2733)

def resetAll():
    for i in range(len(playersList)):
        playersList[i].resetAll()
    

In [ ]:
n = len(playersList)
playerComboGrid = [[0 for i in range(n)] for j in range(n)]

for i in range(n):
    for j in range(n):
        expected = expected_score(playersList[i], playersList[j])
        p_draw = draw_chance(playersList[i], playersList[j])
        p_win = expected - (p_draw * 0.5)
        playerComboGrid[i][j] = (p_win, p_draw, expected)
  

In [ ]:
def updateRating(self):
        sI = self.index
        rating = self.rating
        
        for W in self.winList:
            oI = W.index
            # expectedPerf = expected_score(self.init_rating, playersList[W].init_rating)
            rating += 10 * (1 - playerComboGrid[sI][oI][2])
        for L in self.lossList:
            oI = L.index 
            # expectedPerf = expected_score(self.init_rating, playersList[L].init_rating)
            rating += 10 * (0 - playerComboGrid[oI][sI][2])
        for D in self.drawList:
            oI = D.index
            # expectedPerf = expected_score(self.init_rating, playersList[D].init_rating)
            rating += 10 * (0.5 - playerComboGrid[sI][oI][2])
            
        self.rating = rating

# sI is the self index, oI is the opponents' index (referring to index in playersList)
            
Player.updateRating = updateRating 

In [ ]:
def simulate_game(A,B):
    A.colorsList.append(1)
    B.colorsList.append(2)
    A.opponentsList.append(B.index)
    B.opponentsList.append(A.index)
    p_a_win = playerComboGrid[A.index][B.index][0]
    p_draw = playerComboGrid[A.index][B.index][1]
    roll = rand.random()

    if roll < 1 - p_a_win - p_draw:
        # A.addLoss(B)
        A.losses += 1
        # A.gameResults.append("L")
        A.lossList.append(B)

        # B.addWin(A)
        B.wins += 1
        # B.gameResults.append("W")
        B.winList.append(A)

        # A.updateRating(ratingB, "L")
        # B.updateRating(ratingA, "W")
    elif roll < 1 - p_draw:
        # A.addWin(B)
        A.wins +=1
        # A.gameResults.append("W")
        A.winList.append(B)

        # B.addLoss(A)
        B.losses += 1
        # B.gameResults.append("L")
        B.lossList.append(A)
        
        # A.updateRating(ratingB, "W")
        # B.updateRating(ratingA, "L")
    else:
        # A.addDraw(B)
        A.draws += 1
        # A.gameResults.append("D")
        A.drawList.append(B)

        # B.addDraw(A)
        B.draws +=1
        # B.gameResults.append("D")
        B.drawList.append(A)


        # A.updateRating(ratingB, "D")
        # B.updateRating(ratingA, "D")
       
def simulate_round_robin(players, rounds):
    for k in range(rounds):
        for i in range(len(players)):
            for j in range(i+1, len(players)):
                roll = rand.random()
                if roll < 0.5:
                    simulate_game(players[i],players[j])
                else:
                    simulate_game(players[j],players[i])

def simulate_tiebreak_game(A,B):
    q = playerComboGrid[A.index][B.index][0]
    roll = rand.random()
    if roll >= q:
        A.losses += 1
        # A.gameResults.append("L")
        A.lossList.append(B)

        B.wins += 1
        # B.gameResults.append("W")
        B.winList.append(A)
        return B
    A.wins +=1
    # A.gameResults.append("W")
    A.winList.append(B)
    
    B.losses += 1
    # B.gameResults.append("L")
    B.lossList.append(A)



def simulate_knockout(players, rounds):
    winners = []
    current = players.copy()
    
    while len(current) > 1:
        result = False
        roll = math.floor(1 + rand.random()*(len(current)-1))

        while result == False:
            for i in range (rounds):
                simulate_game(current[0], current[roll])
                simulate_game(current[roll], current[0])
            current[0].setScore()
            current[roll].setScore()
            if current[0].score > 2:
                winners.append(current[0])
                result = True
            elif current[roll].score > 2:
                winners.append(current[roll])
                result = True
            current[0].resetAll()
            current[roll].resetAll()

        current.pop(roll)
        current.pop(0)
        if len(current) == 0 and len(winners) > 1:
            current = winners.copy()
            winners.clear()

    tournamentWinnersList[winners[0].index] += 1


In [ ]:
def orderPlayers(players):
    names = []
    
    for player in players:
        scoresList.append(player.score)
        tiebreaksList.append(player.tiebreak)
        names.append(player.name)
    
    npScores = np.array(scoresList)
    npTiebreaks = np.array(tiebreaksList)
    npPlayers = np.array(players)
    npNames = np.array(names)

    indices = np.argsort(npTiebreaks)
    pre_sorted_scores = npScores[indices]
    final_indices = indices[np.argsort(pre_sorted_scores)]

    npScores = npScores[final_indices]
    npTiebreaks = npTiebreaks[final_indices] 
    npPlayers = npPlayers[final_indices]
    npNames = npNames[final_indices]

    for i in range (len(npPlayers)):
        npPlayers[i].tournamentResults.append(10-i)
    winnersList.append(npNames[-1])
    tournamentWinnersList[npPlayers[-1].index] += 1
    
    scoresList.clear()
    tiebreaksList.clear()


In [ ]:
def simulateTournaments(count, format, players):
    if format == "RR": # round robin
        for i in range (count):
            resetAll()
            simulate_round_robin(players, 1)

            for i in range(len(players)):
                players[i].setScore()
                players[i].updateRating()

            for i in range(len(players)):
                players[i].calculateTiebreak()

            orderPlayers(players)

    elif format == "KO": #knock out
        for i in range (count):
            resetAll()
            simulate_knockout(players, 2)

            # for i in range(len(players)):
            #     players[i].updateRating()
            #     playersList[i].calculateTiebreak()
                

In [ ]:
simulateTournaments(1000000, "KO", playersList)

# for i in range(len(playersList)):
#     print(playersList[i].name)
#     print(playersList[i].getRecordString())
#     print(playersList[i].getScoreString()) 
#     print(playersList[i].rating)
#     print(playersList[i].tiebreak)

print(tournamentWinnersList)

In [ ]:
playerLabels = []
for i in range(len(playersList)):
    playerLabels.append(playersList[i].name)
print(playerLabels)
print(tournamentWinnersList)
plt.pie(tournamentWinnersList, labels=playerLabels, autopct='%1.1f%%')
plt.title("UzChess Cup 2026")
plt.show()

In [ ]:
#todo - fix the RR pairing system - gives 35 free elo to players who play white a lot

In [ ]:
import cProfile

cProfile.run("simulateTournaments(10000, 'RR', playersList)")